In [0]:
dbutils.widgets.text("catalog","sl")
catalog = dbutils.widgets.get("catalog")

#passenger

In [0]:
spark.sql(f"""
       create table if not exists {catalog}.bronze.passenger  COMMENT 'it contains passenger details.'        
""")

In [0]:
spark.sql(f"""
COPY INTO {catalog}.bronze.passenger
FROM (
    SELECT *,
    "251801744149683" as pipeline_job_id,
    _metadata.file_path as source_path,
    current_timestamp() as ingested_at
    FROM "/Volumes/{catalog}/landing/raw/passengers/"
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true')      
COPY_OPTIONS ('mergeSchema' = 'true')
""")

In [0]:
spark.table(f"{catalog}.bronze.passenger").count()

#booking

In [0]:
spark.sql(f"""
       create table if not exists {catalog}.bronze.booking  COMMENT 'it contains passenger booking details.'        
""")

In [0]:
spark.sql(f"""
COPY INTO {catalog}.bronze.booking
FROM (
    SELECT *,
    "251801744149683" as pipeline_job_id,
    _metadata.file_path as source_path,
    current_timestamp() as ingested_at
    FROM "/Volumes/{catalog}/landing/raw/bookings/"
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true')      
COPY_OPTIONS ('mergeSchema' = 'true')
""")

In [0]:
spark.table(f"{catalog}.bronze.booking").count()

#flight_booking

In [0]:
from pyspark.sql.functions import *

spark.sql(f"""
       create table if not exists {catalog}.bronze.flight_booking  COMMENT 'it contains flight booking details.'        
""")

df = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("cloudFiles.schemaLocation", f"/Volumes/{catalog}/bronze/raw/schema/flight_booking") \
    .option("cloudFiles.inferColumnTypes", "true") \
    .option("header", "true") \
    .load(f"/Volumes/{catalog}/landing/raw/flight_bookings/")

df = df.withColumn("pipeline_run_id", lit("251801744149683")) \
    .withColumn("source_path", col("_metadata.file_path")) \
    .withColumn("ingested_at", current_timestamp())

df.writeStream \
    .option("checkpointLocation", f"/Volumes/{catalog}/bronze/raw/checkpoints/flight_booking") \
    .option("mergeSchema", "true") \
    .trigger(availableNow=True) \
    .toTable(f"{catalog}.bronze.flight_booking")

In [0]:
spark.table(f"{catalog}.bronze.flight_booking").count()

#airline

In [0]:
from pyspark.sql.functions import *

df = spark.read \
    .option("header","true") \
    .csv(f"/Volumes/{catalog}/landing/raw/airline.csv")

df = df.withColumn("pipeline_run_id", lit("251801744149683")) \
    .withColumn("source_path",col("_metadata.file_path")) \
    .withColumn("ingested_at",current_timestamp()) 

df = df.write \
    .option("mergeSchema","true") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.bronze.airline")

In [0]:
spark.sql(f"""
          COMMENT ON TABLE {catalog}.bronze.airline IS 'IT IS A AIRLINE MASTER TABLE'
""")

In [0]:
spark.table(f"{catalog}.bronze.airline").count()

#route

In [0]:
from pyspark.sql.functions import *

df = spark.read \
    .option("header","true") \
    .csv(f"/Volumes/{catalog}/landing/raw/route.csv")

df = df.withColumn("pipeline_run_id", lit("251801744149683")) \
.withColumn("source_path",col("_metadata.file_path")) \
    .withColumn("ingested_at",current_timestamp()) 

df = df.write \
    .option("mergeSchema","true") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.bronze.route")

In [0]:
spark.table(f"{catalog}.bronze.route").count()

In [0]:
spark.sql(f"""
          COMMENT ON TABLE {catalog}.bronze.route IS 'IT contains  booking line item.'
""")

#airline_route

In [0]:
from pyspark.sql.functions import *

df = spark.read \
    .option("header","true") \
    .csv(f"/Volumes/{catalog}/landing/raw/airline_route.csv")

df = df.withColumn("pipeline_run_id", lit("251801744149683")) \
.withColumn("source_path",col("_metadata.file_path")) \
    .withColumn("ingested_at",current_timestamp()) 

df = df.write \
    .option("mergeSchema","true") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.bronze.airline_route")
    

In [0]:
spark.table(f"{catalog}.bronze.airline_route").count()

In [0]:
spark.sql(f"""
          COMMENT ON TABLE {catalog}.bronze.airline_route IS 'IT contains  airline to route data.'
""")

#nation

In [0]:
from pyspark.sql.functions import *

df = spark.read \
    .option("header","true") \
    .csv(f"/Volumes/{catalog}/landing/raw/nation.csv")

df = df.withColumn("pipeline_run_id",lit("251801744149683")) \
.withColumn("source_path",col("_metadata.file_path")) \
    .withColumn("ingested_at",current_timestamp()) 

df = df.write \
    .option("mergeSchema","true") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.bronze.nation")
    

In [0]:
spark.sql(f"""
          COMMENT ON TABLE {catalog}.bronze.nation IS 'IT contains nation master wit parent region mapping.'
""")

In [0]:
spark.table(f"{catalog}.bronze.nation").count()

#region

In [0]:
from pyspark.sql.functions import *

df = spark.read \
    .option("header","true") \
    .csv(f"/Volumes/{catalog}/landing/raw/region.csv")

df = df.withColumn("pipeline_run_id",lit("251801744149683")) \
.withColumn("source_path",col("_metadata.file_path")) \
    .withColumn("ingested_at",current_timestamp()) 

df = df.write \
    .option("mergeSchema","true") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.bronze.region")
    

In [0]:
spark.sql(f"""
          COMMENT ON TABLE {catalog}.bronze.region IS 'IT contains geographic region master.'
""")

In [0]:
spark.table(f"{catalog}.bronze.region").count()